# Actividad 4. Selección de modelos y prototipado rápido
## Auditoría de conjuntos de datos para reconocimiento de placas vehiculares
**Continuidad:** Fase I – Business Case – Unidad Habitacional TecMilenio.  
**Objetivo:** verificar integridad, estructura, etiquetas, duplicados y adecuación de fuentes públicas antes de seleccionar un conjunto de evaluación.  
**Glosario:** OCR: *Optical Character Recognition* (reconocimiento óptico de caracteres); CPU: *Central Processing Unit* (unidad central de procesamiento); GPU: *Graphics Processing Unit* (unidad de procesamiento gráfico); CSV: *Comma-Separated Values* (valores separados por comas); SHA-256: *Secure Hash Algorithm 256* (algoritmo criptográfico de resumen de 256 bits).  
**Entorno:** Google Colab, CPU suficiente. **No se realiza entrenamiento ni se ejecutan modelos OCR en esta auditoría.**

### Criterios de calidad y trazabilidad
Se consideran fuentes candidatas *sonnetechnology/license-plate-text-recognition-full* y *prithivMLmods/Number-Plate-Recognition*. Su elección definitiva dependerá de la evidencia obtenida. Las etiquetas automáticas no se considerarán verdad de referencia sin verificación. La coincidencia de texto entre registros tampoco demuestra por sí sola duplicación de imágenes.  
**Nota de alcance:** la auditoría automática identifica posibles errores; la revisión visual y la verificación manual de las transcripciones siguen siendo necesarias. Un conjunto con matrículas de otras regiones no sustituye la evaluación con matrículas mexicanas para un despliegue real.

In [1]:
# CELDA 1 — Preparación del entorno (CPU)
# Instalar las librerías necesarias para acceder a datasets y analizar imágenes.
%pip -q install -U "datasets>=3.0,<5" "huggingface_hub>=0.28" pillow pandas matplotlib tqdm
print("Instalación terminada. Si Colab solicita reiniciar la sesión, hacerlo antes de continuar.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 3.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.
Instalación terminada. Si Colab solicita reiniciar la sesión, hacerlo antes de continuar.


In [2]:
# CELDA 2 — Importaciones y configuración reproducible
import re, io, os, json, time, hashlib, random
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from datasets import load_dataset, get_dataset_config_names
random.seed(42)
DIRECTORIO = Path('/content/auditoria_ocr')
DIRECTORIO.mkdir(parents=True, exist_ok=True)
# Auditar primero una muestra controlada; incrementar solo cuando hayamos validado esquemas y tiempos.
LIMITE_POR_FUENTE = 150
FUENTES = {
    'sonne': 'sonnetechnology/license-plate-text-recognition-full',
    'number_plate': 'prithivMLmods/Number-Plate-Recognition'
}
print('Carpeta de resultados:', DIRECTORIO)
print('Límite de inspección por fuente:', LIMITE_POR_FUENTE)

Carpeta de resultados: /content/auditoria_ocr
Límite de inspección por fuente: 150


In [3]:
# CELDA 3 — Carga controlada y revisión de esquemas
# stream=True evita descargar necesariamente toda la colección; algunos repositorios requieren ajustes.
registros = {}
estados = []
for alias, identificador in FUENTES.items():
    inicio = time.perf_counter()
    try:
        ds = load_dataset(identificador, split='train', streaming=True)
        ejemplos = list(ds.take(LIMITE_POR_FUENTE))
        registros[alias] = ejemplos
        estados.append({'fuente': alias, 'id': identificador, 'registros_cargados':len(ejemplos),
                        'columnas': ', '.join(ejemplos[0].keys()) if ejemplos else 'sin datos',
                        'segundos': round(time.perf_counter()-inicio,2), 'estado': 'OK'})
    except Exception as e:
        registros[alias] = []
        estados.append({'fuente':alias, 'id':identificador, 'registros_cargados':0,
                        'columnas':'', 'segundos':round(time.perf_counter()-inicio,2),
                        'estado':f'{type(e).__name__}: {str(e)[:220]}'})
display(pd.DataFrame(estados))
print('IMPORTANTE: la muestra cargada no equivale al total declarado del dataset.')

README.md:   0%|          | 0.00/1.63k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

         fuente                                                 id  \
0         sonne  sonnetechnology/license-plate-text-recognition...   
1  number_plate             prithivMLmods/Number-Plate-Recognition   

   registros_cargados                                      columnas  segundos  \
0                 150  image, image_id, width, height, bbox, target     14.08   
1                 150                               image, response      5.69   

  estado  
0     OK  
1     OK  

IMPORTANTE: la muestra cargada no equivale al total declarado del dataset.


In [4]:
# CELDA 4 — Inspección segura de campos y tipos
# Mostrar solo resumen de campos, sin imprimir imágenes ni exponer datos masivos.
for fuente, filas in registros.items():
    print(f'\n{fuente}: {len(filas)} registros en la muestra')
    for f in filas[:2]:
        print({k: (f'<Imagen PIL {v.size}>' if isinstance(v, Image.Image) else
                   f'<bytes {len(v)}>' if isinstance(v, bytes) else
                   str(v)[:180]) for k,v in f.items()})


sonne: 150 registros en la muestra
{'image': '<Imagen PIL (1024, 682)>', 'image_id': '487', 'width': '1024', 'height': '682', 'bbox': '[[118.0, 584.0, 203.76000213623047, 616.658935546875], [488.0, 506.0, 510.3999996185303, 514.9655723571777]]', 'target': "['X78IAEN', 'SUBT']"}
{'image': '<Imagen PIL (472, 303)>', 'image_id': '1502', 'width': '472', 'height': '303', 'bbox': '[[174.0, 147.0, 227.0, 165.0]]', 'target': "['TOTAL: 853.25']"}

number_plate: 150 registros en la muestra
{'image': '<Imagen PIL (1920, 1080)>', 'response': '```json\n{\n  "vehicle_color": "black",\n  "license_plate_number": "77-ZFK-8"\n}\n```'}
{'image': '<Imagen PIL (3112, 2340)>', 'response': '```json\n{\n  "vehicle_color": "yellow",\n  "license_plate_number": "42-UK-32"\n}\n```'}


In [5]:
# CELDA 5 — Extracción adaptable de imágenes y anotaciones
# Esquemas distintos requieren tratamiento distinto. No inventamos etiquetas faltantes.
def como_pil(valor):
    try:
        if isinstance(valor, Image.Image): return ImageOps.exif_transpose(valor).convert('RGB')
        if isinstance(valor, dict):
            if valor.get('bytes'): return Image.open(io.BytesIO(valor['bytes'])).convert('RGB')
            if valor.get('path') and os.path.isfile(valor['path']): return Image.open(valor['path']).convert('RGB')
    except Exception: pass
    return None

def primer_texto(row):
    for k in ('license_plate_number','plate','plate_number','text','label','target','transcription'):
        if k in row and row[k] is not None:
            v=row[k]
            if isinstance(v, str): return v, k
            if isinstance(v, list): return ' | '.join(map(str,v)),k
    return '', 'no_identificada'

def obtener_imagen(row):
    for k in ('image','img','picture','image_file'):
        if k in row:
            img=como_pil(row[k])
            if img is not None:return img,k
    return None,'no_identificada'

indice=[]
imagenes={}
for fuente, filas in registros.items():
    for i,r in enumerate(filas):
        im,campo_im=obtener_imagen(r)
        etiqueta,campo_et=primer_texto(r)
        key=f'{fuente}_{i:05d}'
        if im is not None:
            # Hash del contenido visual normalizado: detecta imágenes idénticas aunque cambie el archivo.
            h=hashlib.sha256(im.tobytes()+str(im.size).encode()).hexdigest()
            imagenes[key]=im
            w,hg=im.size
        else: h,w,hg=None,None,None
        indice.append({'id':key,'fuente':fuente,'campo_imagen':campo_im,
                       'campo_etiqueta':campo_et,'texto_original':etiqueta,
                       'ancho':w,'alto':hg,'hash_imagen':h,
                       'imagen_legible':im is not None,
                       'bbox_disponible':any(k in r for k in ('bbox','boxes','bounding_box','annotations'))})
auditoria=pd.DataFrame(indice)
display(auditoria.groupby('fuente',dropna=False).agg(
    registros=('id','count'), imagenes_legibles=('imagen_legible','sum'),
    con_bbox=('bbox_disponible','sum')).reset_index() if len(auditoria) else pd.DataFrame())

         fuente  registros  imagenes_legibles  con_bbox
0  number_plate        150                150         0
1         sonne        150                150       150

In [6]:
# CELDA 6 — Detección de alertas y duplicados exactos
# Este filtro es exploratorio; patrones de placas válidos varían entre países y entidades.
if len(auditoria):
    auditoria['texto_normalizado']=auditoria['texto_original'].fillna('').astype(str).str.upper().str.strip()
    auditoria['texto_vacio']=auditoria['texto_normalizado'].eq('')
    auditoria['texto_sospechoso']=auditoria['texto_normalizado'].str.contains(
        r'TOTAL|INVOICE|RECEIPT|PRICE|\*{2,}|\||\n',regex=True,na=False)
    auditoria['duplicado_imagen_exacto']=auditoria['hash_imagen'].notna() & auditoria['hash_imagen'].duplicated(keep=False)
    auditoria['etiqueta_repetida']=auditoria['texto_normalizado'].ne('') & auditoria.duplicated(
        subset=['fuente','texto_normalizado'],keep=False)
    auditoria['requiere_revision']=(~auditoria['imagen_legible'] | auditoria['texto_vacio'] |
        auditoria['texto_sospechoso'] | auditoria['duplicado_imagen_exacto'])
    resumen=auditoria.groupby('fuente').agg(
        registros=('id','size'), sin_imagen=('imagen_legible',lambda s:int((~s).sum())),
        texto_vacio=('texto_vacio','sum'), texto_sospechoso=('texto_sospechoso','sum'),
        duplicado_exacto=('duplicado_imagen_exacto','sum'), etiqueta_repetida=('etiqueta_repetida','sum'),
        revision=('requiere_revision','sum')).reset_index()
    display(resumen)
else: print('No hay registros. Revisar los errores de la CELDA 3.')
print('Nota: una etiqueta repetida NO equivale a una imagen duplicada.')

         fuente  registros  sin_imagen  texto_vacio  texto_sospechoso  \
0  number_plate        150           0          150                 0   
1         sonne        150           0            0                10   

   duplicado_exacto  etiqueta_repetida  revision  
0                 0                  0       150  
1                 0                 38        10  

Nota: una etiqueta repetida NO equivale a una imagen duplicada.


In [7]:
# CELDA 7 — Muestra visual para revisión humana
# Revisar si la imagen es placa completa, recorte correcto, o fotografía de un vehículo/escena.
if imagenes:
    ids=list(imagenes.keys())
    seleccion=random.Random(42).sample(ids,min(16,len(ids)))
    filas=(len(seleccion)+3)//4
    fig,ax=plt.subplots(filas,4,figsize=(16,3.6*filas))
    for a in np.array(ax).reshape(-1): a.axis('off')
    for a,id_ in zip(np.array(ax).reshape(-1),seleccion):
        a.imshow(imagenes[id_]);a.axis('off')
        texto=auditoria.loc[auditoria.id.eq(id_),'texto_original'].iloc[0]
        a.set_title(f'{id_}\nEtiqueta: {str(texto)[:35]}',fontsize=9)
    plt.tight_layout();plt.show()
else: print('No hay imágenes para visualizar.')

<Figure size 1600x1440 with 16 Axes>

In [8]:
# CELDA 8 — Plantilla de verificación manual
# No se asignará automáticamente 'correcto': esa decisión exige contrastar la imagen con el texto.
if len(auditoria):
    revision=auditoria[['id','fuente','texto_original','imagen_legible','texto_sospechoso',
                        'duplicado_imagen_exacto','requiere_revision']].copy()
    revision['es_placa_visible']='PENDIENTE'  # SI / NO
    revision['placa_recortada']='PENDIENTE' # SI / NO
    revision['transcripcion_verificada']=''  # introducir caracteres visibles
    revision['aprobado_para_evaluacion']='PENDIENTE' # SI / NO
    revision['motivo_exclusion']=''
    ruta=DIRECTORIO/'revision_manual_plantilla.csv'
    revision.to_csv(ruta,index=False,encoding='utf-8-sig')
    print('Plantilla creada:',ruta)
    display(revision.head(10))
else: print('No se generó plantilla porque no hay registros.')

Plantilla creada: /content/auditoria_ocr/revision_manual_plantilla.csv


            id fuente  texto_original  imagen_legible  texto_sospechoso  \
0  sonne_00000  sonne  X78IAEN | SUBT            True              True   
1  sonne_00001  sonne   TOTAL: 853.25            True              True   
2  sonne_00002  sonne             SR:            True             False   
3  sonne_00003  sonne      51F-826.06            True             False   
4  sonne_00004  sonne        ST#02928            True             False   
5  sonne_00005  sonne             ***            True              True   
6  sonne_00006  sonne      51F-153.80            True             False   
7  sonne_00007  sonne              SR            True             False   
8  sonne_00008  sonne           82829            True             False   
9  sonne_00009  sonne      48A-051.77            True             False   

   duplicado_imagen_exacto  requiere_revision es_placa_visible  \
0                    False               True        PENDIENTE   
1                    False               T

In [9]:
# CELDA 9 — Exportación de la auditoría preliminar
# Solo resultados efectivamente observados: NO equivale a dataset final aprobado.
if len(auditoria):
    ruta_a=DIRECTORIO/'auditoria_preliminar.csv'
    ruta_r=DIRECTORIO/'resumen_auditoria.csv'
    auditoria.drop(columns=['hash_imagen']).to_csv(ruta_a,index=False,encoding='utf-8-sig')
    resumen.to_csv(ruta_r,index=False,encoding='utf-8-sig')
    print('Archivos exportados:',ruta_a,ruta_r,sep='\n - ')
    print('Cantidad inspeccionada:',len(auditoria),'(no extrapolar tasas al total sin muestreo representativo)')
else: print('Sin resultados que exportar.')

Archivos exportados:
 - /content/auditoria_ocr/auditoria_preliminar.csv
 - /content/auditoria_ocr/resumen_auditoria.csv
Cantidad inspeccionada: 300 (no extrapolar tasas al total sin muestreo representativo)


### Dictamen y siguientes decisiones
**Estado por defecto: decisión pendiente.** Antes de seleccionar definitivamente un dataset, verificar imágenes, coordenadas y etiquetas; determinar si hay matrículas duplicadas; documentar descartes. Para el experimento final se requiere una muestra de placas recortadas con transcripciones humanas verificadas y una política clara de exclusión.  
**Separación metodológica:** este notebook no ejecuta reconocimiento OCR ni calcula rendimiento de modelos. La comparación de los tres modelos se desarrollará en otro notebook y con un conjunto de prueba común.  
**Fuentes:** [Sonne Technology](https://huggingface.co/datasets/sonnetechnology/license-plate-text-recognition-full) y [Number Plate Recognition](https://huggingface.co/datasets/prithivMLmods/Number-Plate-Recognition). Revalidar sus condiciones de uso antes de publicar imágenes en GitHub.  
**Limitación:** la ejecución real en Google Colab es necesaria para confirmar estructura, disponibilidad y calidad; ningún valor de salida del código se considera observado hasta ejecutar las celdas.

### Hallazgo de auditoría y control de cambios

En la primera auditoría se detectaron 150 etiquetas aparentemente vacías en la fuente **Number Plate Recognition**. La inspección del esquema reveló que la matrícula estaba anidada dentro de `response`, como un objeto JSON (*JavaScript Object Notation*, formato de intercambio de datos). Por ello se agregó la CELDA 10 para corregir la extracción **sin alterar la evidencia de la primera ejecución**. Las cantidades definitivas dependen de volver a ejecutar esa celda y verificar su salida.

La revisión visual inicial también mostró fotografías completas de vehículos, no siempre imágenes de matrículas recortadas. Antes de evaluar los modelos de reconocimiento óptico de caracteres (OCR, *Optical Character Recognition*) será necesario validar coordenadas y generar recortes verificables.

**Dictamen actual:** ninguno de los conjuntos de datos está aprobado todavía como referencia de evaluación. No se extrapolan las tasas observadas en las muestras al total de los repositorios.


In [10]:
# CELDA 10 — Corrección de etiquetas JSON y contraste de auditoría
# El campo response de Number Plate Recognition contiene un JSON (JavaScript Object Notation)
# con la matrícula en license_plate_number; NO debe considerarse vacío.
# Se conserva el diagnóstico anterior como evidencia y se recalculan los indicadores.

def extraer_response(fila):
    valor = fila.get("response", "")
    if isinstance(valor, dict):
        obj = valor
    elif isinstance(valor, str):
        # Algunos registros contienen JSON dentro de un bloque de código Markdown.
        texto = re.sub(r"^\s*```(?:json)?\s*|\s*```\s*$", "", valor.strip(), flags=re.I).strip()
        try:
            obj = json.loads(texto)
        except (json.JSONDecodeError, TypeError):
            return "", "response_json_no_interpretable"
    else:
        return "", "response_formato_desconocido"
    placa = obj.get("license_plate_number", "")
    if not isinstance(placa, str):
        return "", "license_plate_number_no_texto"
    return placa.strip(), "response.license_plate_number"

correcciones = []
for i, fila in enumerate(registros.get("number_plate", [])):
    identificador = f"number_plate_{i:05d}"
    placa, origen = extraer_response(fila)
    mascara = auditoria["id"].eq(identificador)
    auditoria.loc[mascara, "texto_original"] = placa
    auditoria.loc[mascara, "campo_etiqueta"] = origen
    correcciones.append({"id": identificador, "etiqueta_extraida": placa, "origen": origen})

display(pd.DataFrame(correcciones).groupby("origen", dropna=False).agg(
    registros=("id", "size"),
    placas_no_vacias=("etiqueta_extraida", lambda s: int(s.astype(str).str.strip().ne("").sum()))
).reset_index())

# Recalcular las alertas tras corregir las etiquetas.
auditoria["texto_normalizado"] = auditoria["texto_original"].fillna("").astype(str).str.upper().str.strip()
auditoria["texto_vacio"] = auditoria["texto_normalizado"].eq("")
auditoria["texto_sospechoso"] = auditoria["texto_normalizado"].str.contains(
    r"TOTAL|INVOICE|RECEIPT|PRICE|\*{2,}|\||\n", regex=True, na=False)
auditoria["duplicado_imagen_exacto"] = auditoria["hash_imagen"].notna() & auditoria["hash_imagen"].duplicated(keep=False)
auditoria["etiqueta_repetida"] = auditoria["texto_normalizado"].ne("") & auditoria.duplicated(
    subset=["fuente", "texto_normalizado"], keep=False)
auditoria["requiere_revision"] = (
    ~auditoria["imagen_legible"] | auditoria["texto_vacio"]
    | auditoria["texto_sospechoso"] | auditoria["duplicado_imagen_exacto"]
)
resumen_corregido = auditoria.groupby("fuente").agg(
    registros=("id", "size"),
    sin_imagen=("imagen_legible", lambda s: int((~s).sum())),
    texto_vacio=("texto_vacio", "sum"),
    texto_sospechoso=("texto_sospechoso", "sum"),
    duplicado_exacto=("duplicado_imagen_exacto", "sum"),
    etiqueta_repetida=("etiqueta_repetida", "sum"),
    revision=("requiere_revision", "sum")
).reset_index()
display(resumen_corregido)
print("IMPORTANTE: son alertas exploratorias, no una certificación de placas válidas.")


                          origen  registros  placas_no_vacias
0  response.license_plate_number        150               150

         fuente  registros  sin_imagen  texto_vacio  texto_sospechoso  \
0  number_plate        150           0            0                 0   
1         sonne        150           0            0                10   

   duplicado_exacto  etiqueta_repetida  revision  
0                 0                 13         0  
1                 0                 38        10  

IMPORTANTE: son alertas exploratorias, no una certificación de placas válidas.


In [11]:
# CELDA 11 — Exportación de auditoría corregida
# Conserva la exportación inicial por separado para mantener la trazabilidad.
ruta_detalle = DIRECTORIO / 'auditoria_corregida.csv'
ruta_resumen = DIRECTORIO / 'resumen_auditoria_corregida.csv'
auditoria.drop(columns=['hash_imagen'], errors='ignore').to_csv(ruta_detalle, index=False, encoding='utf-8-sig')
resumen_corregido.to_csv(ruta_resumen, index=False, encoding='utf-8-sig')
print('Resultados corregidos exportados:')
print(ruta_detalle)
print(ruta_resumen)
print('Próximo control: revisar los recortes de placas antes de aprobar el dataset.')


Resultados corregidos exportados:
/content/auditoria_ocr/auditoria_corregida.csv
/content/auditoria_ocr/resumen_auditoria_corregida.csv
Próximo control: revisar los recortes de placas antes de aprobar el dataset.


In [12]:

# CELDA 12 — Validación visual de recortes de placas
# Actividad 4. Selección de modelos y prototipado rápido
#
# Objetivo:
# Verificar si las coordenadas del dataset Sonne Technology
# permiten extraer correctamente las matrículas vehiculares.
#
# OCR = Optical Character Recognition
# (Reconocimiento óptico de caracteres).
# bbox = Bounding Box (caja delimitadora).
#
# IMPORTANTE:
# Una coordenada válida no demuestra que la placa sea legible
# ni que la etiqueta sea correcta. Se requiere revisión humana.

import ast
import json
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# -------------------------------------------------------
# 1. Recuperar registros e imágenes ya cargados
# -------------------------------------------------------

fuente = "sonne"
filas_sonne = registros.get(fuente, [])

if not filas_sonne:
    raise ValueError(
        "No hay registros Sonne cargados. "
        "Ejecuta primero la CELDA 3."
    )

# -------------------------------------------------------
# 2. Interpretar listas de coordenadas y etiquetas
# -------------------------------------------------------

def interpretar_lista(valor):
    if isinstance(valor, (list, tuple)):
        return list(valor)

    if isinstance(valor, str):
        try:
            resultado = json.loads(valor)
        except (json.JSONDecodeError, TypeError):
            try:
                resultado = ast.literal_eval(valor)
            except (ValueError, SyntaxError):
                return []
        return resultado if isinstance(resultado, list) else []

    return []


def coordenadas_validas(box, ancho, alto):
    try:
        if not isinstance(box, (list, tuple)) or len(box) != 4:
            return False

        x1, y1, x2, y2 = map(float, box)

        if not all(np.isfinite([x1, y1, x2, y2])):
            return False

        return (
            0 <= x1 < x2 <= ancho
            and 0 <= y1 < y2 <= alto
        )
    except (TypeError, ValueError):
        return False

# -------------------------------------------------------
# 3. Generar recortes verificables
# -------------------------------------------------------

registros_recortes = []
recortes_validos = {}

for i, fila in enumerate(filas_sonne):

    identificador = f"sonne_{i:05d}"
    imagen = imagenes.get(identificador)

    if imagen is None:
        registros_recortes.append({
            "id": identificador,
            "region": None,
            "etiqueta": "",
            "estado": "imagen_no_disponible"
        })
        continue

    cajas = interpretar_lista(fila.get("bbox", []))
    etiquetas = interpretar_lista(fila.get("target", []))

    # Las regiones y etiquetas deben corresponderse.
    if len(cajas) != len(etiquetas):
        registros_recortes.append({
            "id": identificador,
            "region": None,
            "etiqueta": "",
            "estado": "cantidad_etiquetas_no_coincide"
        })
        continue

    ancho, alto = imagen.size

    for j, (caja, etiqueta) in enumerate(zip(cajas, etiquetas)):

        estado = "coordenadas_validas"

        if not coordenadas_validas(caja, ancho, alto):
            estado = "coordenadas_invalidas"

        registro = {
            "id": identificador,
            "region": j,
            "etiqueta": str(etiqueta),
            "estado": estado
        }

        registros_recortes.append(registro)

        if estado == "coordenadas_validas":
            x1, y1, x2, y2 = map(float, caja)

            # floor/ceil evita perder píxeles en los bordes.
            recorte = imagen.crop((
                math.floor(x1),
                math.floor(y1),
                math.ceil(x2),
                math.ceil(y2)
            ))

            clave = f"{identificador}_region_{j}"
            recortes_validos[clave] = {
                "imagen": recorte,
                "etiqueta": str(etiqueta),
                "id": identificador,
                "region": j
            }

# -------------------------------------------------------
# 4. Resumen cuantitativo
# -------------------------------------------------------

df_recortes = pd.DataFrame(registros_recortes)

print("AUDITORÍA DE RECORTES — SONNE TECHNOLOGY")
print("-" * 55)

print("Imágenes examinadas:", len(filas_sonne))
print("Regiones identificadas:", len(df_recortes))
print("Recortes generados:", len(recortes_validos))

display(
    df_recortes["estado"]
    .value_counts()
    .rename_axis("estado")
    .reset_index(name="cantidad")
)

# -------------------------------------------------------
# 5. Muestra visual reproducible
# -------------------------------------------------------

claves = list(recortes_validos.keys())

random.seed(42)
seleccion = random.sample(
    claves,
    min(20, len(claves))
)

if seleccion:

    columnas = 4
    filas = math.ceil(len(seleccion) / columnas)

    fig, ejes = plt.subplots(
        filas, columnas,
        figsize=(16, 3.5 * filas)
    )

    ejes = np.array(ejes).reshape(-1)

    for ax in ejes:
        ax.axis("off")

    for ax, clave in zip(ejes, seleccion):

        dato = recortes_validos[clave]

        ax.imshow(dato["imagen"])
        ax.set_title(
            f"{clave}\nEtiqueta: {dato['etiqueta']}",
            fontsize=9
        )

    plt.suptitle(
        "Auditoría visual de matrículas recortadas",
        fontsize=14
    )

    plt.tight_layout()
    plt.show()

else:
    print("No se encontraron recortes válidos.")

print(
    "\nIMPORTANTE: Los recortes mostrados aún "
    "no están certificados como correctos."
)


AUDITORÍA DE RECORTES — SONNE TECHNOLOGY
-------------------------------------------------------
Imágenes examinadas: 150
Regiones identificadas: 160
Recortes generados: 160


                estado  cantidad
0  coordenadas_validas       160

<Figure size 1600x1750 with 20 Axes>


IMPORTANTE: Los recortes mostrados aún no están certificados como correctos.


In [13]:

# CELDA 13 — Verificación manual y corrección de etiquetas
# Actividad 4. Selección de modelos y prototipado rápido
#
# Objetivo:
# Validar manualmente los recortes de matrículas generados
# en la CELDA 12 y crear etiquetas de referencia confiables.
#
# CSV = Comma-Separated Values
# (Archivo de valores separados por comas).
#
# IMPORTANTE:
# No se modifica el dataset original.
# Los registros se guardan con su identificador y etiqueta original.
# La corrección se basa únicamente en los caracteres visibles.

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path

# -------------------------------------------------------
# 1. Preparar archivo de validación
# -------------------------------------------------------

ruta_validacion = DIRECTORIO / "validacion_manual_placas.csv"

columnas = [
    "id_recorte",
    "id_imagen",
    "region",
    "etiqueta_original",
    "etiqueta_verificada",
    "estado",
    "observaciones"
]

if ruta_validacion.exists():
    validacion = pd.read_csv(
        ruta_validacion,
        dtype=str,
        keep_default_na=False
    )
    print("Se recuperó una validación anterior.")
else:
    validacion = pd.DataFrame(columns=columnas)
    print("Se creó una nueva tabla de validación.")

# -------------------------------------------------------
# 2. Identificar matrículas pendientes
# -------------------------------------------------------

pendientes = [
    clave for clave in recortes_validos
    if clave not in set(validacion["id_recorte"])
]

print("Recortes totales:", len(recortes_validos))
print("Recortes revisados:", len(validacion))
print("Recortes pendientes:", len(pendientes))

# -------------------------------------------------------
# 3. Crear interfaz de revisión
# -------------------------------------------------------

salida = widgets.Output()

texto_etiqueta = widgets.Text(
    description="Matrícula:",
    placeholder="Escribe la transcripción correcta",
    layout=widgets.Layout(width="450px"),
    style={"description_width": "100px"}
)

observaciones = widgets.Textarea(
    description="Observaciones:",
    placeholder="Opcional: desenfoque, inclinación, caracteres dudosos...",
    layout=widgets.Layout(width="600px", height="75px"),
    style={"description_width": "100px"}
)

boton_validar = widgets.Button(
    description="Validar",
    button_style="success"
)

boton_corregir = widgets.Button(
    description="Corregir",
    button_style="info"
)

boton_descartar = widgets.Button(
    description="Descartar",
    button_style="danger"
)

contador = widgets.HTML()

# -------------------------------------------------------
# 4. Mostrar matrícula actual
# -------------------------------------------------------

def mostrar_actual():

    with salida:
        clear_output(wait=True)

        if not pendientes:
            print("VALIDACIÓN FINALIZADA")
            print("Total de registros revisados:", len(validacion))
            print("Archivo:", ruta_validacion)
            return

        clave = pendientes[0]
        dato = recortes_validos[clave]

        contador.value = (
            f"<b>Revisados:</b> {len(validacion)} "
            f" | <b>Pendientes:</b> {len(pendientes)}"
        )

        print("Identificador:", clave)
        print("Etiqueta original:", dato["etiqueta"])

        display(dato["imagen"].resize((
            dato["imagen"].width * 3,
            dato["imagen"].height * 3
        )))

        texto_etiqueta.value = dato["etiqueta"]
        observaciones.value = ""

# -------------------------------------------------------
# 5. Registrar decisión humana
# -------------------------------------------------------

def registrar_decision(estado):

    global validacion

    if not pendientes:
        return

    clave = pendientes[0]
    dato = recortes_validos[clave]

    etiqueta = texto_etiqueta.value.strip().upper()

    if estado in ("validada", "corregida") and not etiqueta:
        with salida:
            print("ERROR: La matrícula no puede estar vacía.")
        return

    if estado == "validada":
        if etiqueta != str(dato["etiqueta"]).strip().upper():
            with salida:
                print(
                    "La transcripción difiere de la original. "
                    "Utiliza el botón Corregir."
                )
            return

    registro = {
        "id_recorte": clave,
        "id_imagen": dato["id"],
        "region": dato["region"],
        "etiqueta_original": dato["etiqueta"],
        "etiqueta_verificada": (
            etiqueta if estado != "descartada" else ""
        ),
        "estado": estado,
        "observaciones": observaciones.value.strip()
    }

    validacion = pd.concat(
        [validacion, pd.DataFrame([registro])],
        ignore_index=True
    )

    # Guardar inmediatamente cada revisión.
    validacion.to_csv(
        ruta_validacion,
        index=False,
        encoding="utf-8-sig"
    )

    pendientes.pop(0)
    mostrar_actual()

# -------------------------------------------------------
# 6. Asignar acciones a los botones
# -------------------------------------------------------

boton_validar.on_click(
    lambda _: registrar_decision("validada")
)

boton_corregir.on_click(
    lambda _: registrar_decision("corregida")
)

boton_descartar.on_click(
    lambda _: registrar_decision("descartada")
)

# -------------------------------------------------------
# 7. Mostrar interfaz
# -------------------------------------------------------

display(contador)
display(salida)
display(texto_etiqueta)
display(observaciones)

display(
    widgets.HBox([
        boton_validar,
        boton_corregir,
        boton_descartar
    ])
)

mostrar_actual()


Se creó una nueva tabla de validación.
Recortes totales: 160
Recortes revisados: 0
Recortes pendientes: 160


HTML(value='')

Output()

Text(value='', description='Matrícula:', layout=Layout(width='450px'), placeholder='Escribe la transcripción c…

Textarea(value='', description='Observaciones:', layout=Layout(height='75px', width='600px'), placeholder='Opc…

In [14]:

# ============================================================
# CELDA 14 — Clasificación automática de riesgos
# Actividad 4. Selección de modelos y prototipado rápido
# ============================================================
#
# OBJETIVO:
# Evaluar automáticamente los 160 recortes obtenidos en la
# CELDA 12, identificar posibles problemas de calidad y
# priorizar los registros para su revisión humana.
#
# OCR = Optical Character Recognition
#       (Reconocimiento óptico de caracteres).
#
# CSV = Comma-Separated Values
#       (Archivo de valores separados por comas).
#
# IMPORTANTE:
# Los filtros detectan indicios de problemas, pero no
# certifican la exactitud de una matrícula.
# No se alterarán las imágenes ni etiquetas originales.
# ============================================================

import re
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image, ImageFilter

# ------------------------------------------------------------
# 1. Verificar información de la CELDA 12
# ------------------------------------------------------------

if not recortes_validos:
    raise ValueError(
        "No hay recortes disponibles. Ejecuta la CELDA 12."
    )

# ------------------------------------------------------------
# 2. Función para analizar cada imagen
# ------------------------------------------------------------

def analizar_recorte(imagen, etiqueta):

    imagen_rgb = imagen.convert("RGB")
    imagen_gris = imagen.convert("L")

    ancho, alto = imagen.size

    # Medición básica de contraste.
    pixels = np.asarray(imagen_gris, dtype=np.float32)
    contraste = float(pixels.std())

    # Estimación de nitidez mediante diferencias de intensidad.
    # No es una medida definitiva de legibilidad.
    dx = np.diff(pixels, axis=1)
    dy = np.diff(pixels, axis=0)

    nitidez = float(
        (
            np.mean(np.abs(dx)) if dx.size else 0
        ) + (
            np.mean(np.abs(dy)) if dy.size else 0
        )
    )

    texto = str(etiqueta).strip().upper()

    alertas = []

    # --------------------------------------------------------
    # 3. Reglas de revisión
    # --------------------------------------------------------

    if not texto:
        alertas.append("Etiqueta vacía")

    if len(texto) < 5:
        alertas.append("Etiqueta demasiado corta")

    if len(texto) > 15:
        alertas.append("Etiqueta extensa")

    if re.search(r"[^A-Z0-9 .-]", texto):
        alertas.append("Caracteres no habituales")

    if not re.search(r"[A-Z]", texto):
        alertas.append("Sin letras")

    if not re.search(r"[0-9]", texto):
        alertas.append("Sin números")

    if ancho < 80 or alto < 25:
        alertas.append("Resolución reducida")

    if contraste < 25:
        alertas.append("Contraste bajo")

    if nitidez < 12:
        alertas.append("Posible desenfoque")

    # Los umbrales son exploratorios.
    # Pueden generar falsos positivos y negativos.

    # --------------------------------------------------------
    # 4. Clasificación de prioridad
    # --------------------------------------------------------

    cantidad_alertas = len(alertas)

    if cantidad_alertas >= 3:
        prioridad = "ALTA"
    elif cantidad_alertas >= 1:
        prioridad = "MEDIA"
    else:
        prioridad = "BAJA"

    return {
        "ancho": ancho,
        "alto": alto,
        "contraste": round(contraste, 2),
        "nitidez": round(nitidez, 2),
        "cantidad_alertas": cantidad_alertas,
        "alertas": " | ".join(alertas),
        "prioridad": prioridad
    }

# ------------------------------------------------------------
# 5. Ejecutar auditoría sobre todos los recortes
# ------------------------------------------------------------

resultados = []

for clave, dato in recortes_validos.items():

    analisis = analizar_recorte(
        dato["imagen"],
        dato["etiqueta"]
    )

    resultados.append({
        "id_recorte": clave,
        "id_imagen": dato["id"],
        "region": dato["region"],
        "etiqueta_original": dato["etiqueta"],
        **analisis
    })

df_priorizacion = pd.DataFrame(resultados)

# ------------------------------------------------------------
# 6. Generar resumen por nivel de prioridad
# ------------------------------------------------------------

orden_prioridad = ["ALTA", "MEDIA", "BAJA"]

resumen_prioridad = (
    df_priorizacion["prioridad"]
    .value_counts()
    .reindex(orden_prioridad, fill_value=0)
    .rename_axis("prioridad")
    .reset_index(name="cantidad")
)

resumen_prioridad["porcentaje"] = (
    resumen_prioridad["cantidad"]
    / len(df_priorizacion) * 100
).round(2)

print("=" * 55)
print("RESULTADOS DE PRIORIZACIÓN — SONNE TECHNOLOGY")
print("=" * 55)

print("Recortes analizados:", len(df_priorizacion))

display(resumen_prioridad)

# ------------------------------------------------------------
# 7. Mostrar principales causas de alerta
# ------------------------------------------------------------

alertas_individuales = (
    df_priorizacion["alertas"]
    .str.split(" | ", regex=False)
    .explode()
)

alertas_individuales = alertas_individuales[
    alertas_individuales.notna()
    & alertas_individuales.ne("")
]

resumen_alertas = (
    alertas_individuales
    .value_counts()
    .rename_axis("tipo_alerta")
    .reset_index(name="cantidad")
)

print("\nPrincipales alertas detectadas:")
display(resumen_alertas)

# ------------------------------------------------------------
# 8. Mostrar los primeros registros de mayor riesgo
# ------------------------------------------------------------

orden = {"ALTA": 0, "MEDIA": 1, "BAJA": 2}

df_priorizacion["orden"] = (
    df_priorizacion["prioridad"].map(orden)
)

df_priorizacion = (
    df_priorizacion
    .sort_values(
        ["orden", "cantidad_alertas"],
        ascending=[True, False]
    )
    .drop(columns="orden")
    .reset_index(drop=True)
)

print("\nPrimeros 15 registros priorizados:")

display(
    df_priorizacion[
        [
            "id_recorte",
            "etiqueta_original",
            "prioridad",
            "cantidad_alertas",
            "alertas"
        ]
    ].head(15)
)

# ------------------------------------------------------------
# 9. Exportar resultados
# ------------------------------------------------------------

ruta_priorizacion = (
    DIRECTORIO / "priorizacion_recortes_sonne.csv"
)

df_priorizacion.to_csv(
    ruta_priorizacion,
    index=False,
    encoding="utf-8-sig"
)

print("\nArchivo CSV generado:")
print(ruta_priorizacion)

print(
    "\nADVERTENCIA: BAJA prioridad no significa "
    "que la etiqueta haya sido validada."
)


RESULTADOS DE PRIORIZACIÓN — SONNE TECHNOLOGY
Recortes analizados: 160


  prioridad  cantidad  porcentaje
0      ALTA        30       18.75
1     MEDIA        75       46.88
2      BAJA        55       34.38


Principales alertas detectadas:


                tipo_alerta  cantidad
0       Resolución reducida        70
1  Etiqueta demasiado corta        38
2                Sin letras        37
3               Sin números        36
4  Caracteres no habituales        22
5            Contraste bajo         2


Primeros 15 registros priorizados:


              id_recorte etiqueta_original prioridad  cantidad_alertas  \
0   sonne_00063_region_0                 =      ALTA                 5   
1   sonne_00090_region_0                 :      ALTA                 5   
2   sonne_00002_region_0               SR:      ALTA                 4   
3   sonne_00005_region_0               ***      ALTA                 4   
4   sonne_00029_region_0               ***      ALTA                 4   
5   sonne_00042_region_0               SR:      ALTA                 4   
6   sonne_00122_region_1              EASY      ALTA                 4   
7   sonne_00000_region_1              SUBT      ALTA                 3   
8   sonne_00007_region_0                SR      ALTA                 3   
9   sonne_00010_region_0                EE      ALTA                 3   
10  sonne_00019_region_0              8.99      ALTA                 3   
11  sonne_00021_region_0                SR      ALTA                 3   
12  sonne_00058_region_0              


Archivo CSV generado:
/content/auditoria_ocr/priorizacion_recortes_sonne.csv

ADVERTENCIA: BAJA prioridad no significa que la etiqueta haya sido validada.


In [15]:

# ============================================================
# CELDA 15 — Validación manual de matrículas por lotes
# Actividad 4. Selección de modelos y prototipado rápido
# ============================================================
#
# OBJETIVO:
# Revisar matrículas por grupos de 10, comenzando con los
# registros de prioridad BAJA de la CELDA 14.
#
# Cada registro tendrá:
# - Imagen recortada.
# - Etiqueta original.
# - Campo editable para corregir la matrícula.
# - Decisión: validar, corregir o descartar.
#
# CSV = Comma-Separated Values
# (Archivo de valores separados por comas).
#
# IMPORTANTE:
# La prioridad BAJA no certifica la etiqueta.
# Todas las decisiones deberán verificarse visualmente.
# Los cambios se guardan sin alterar el dataset original.
# ============================================================

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path

# ------------------------------------------------------------
# 1. Configuración y recuperación de validaciones anteriores
# ------------------------------------------------------------

ruta_validacion = DIRECTORIO / "validacion_manual_placas.csv"

columnas = [
    "id_recorte",
    "id_imagen",
    "region",
    "etiqueta_original",
    "etiqueta_verificada",
    "estado",
    "observaciones"
]

if ruta_validacion.exists():
    validacion_lotes = pd.read_csv(
        ruta_validacion,
        dtype=str,
        keep_default_na=False
    )
else:
    validacion_lotes = pd.DataFrame(columns=columnas)

# Eliminar decisiones duplicadas, conservando la última.
validacion_lotes = validacion_lotes.drop_duplicates(
    subset=["id_recorte"],
    keep="last"
).reset_index(drop=True)

# ------------------------------------------------------------
# 2. Preparar los registros pendientes
# ------------------------------------------------------------

orden_prioridad = ["BAJA", "MEDIA", "ALTA"]

df_pendientes = df_priorizacion.copy()

df_pendientes["orden"] = pd.Categorical(
    df_pendientes["prioridad"],
    categories=orden_prioridad,
    ordered=True
)

df_pendientes = df_pendientes.sort_values(
    ["orden", "id_recorte"]
)

revisados = set(validacion_lotes["id_recorte"])

df_pendientes = df_pendientes[
    ~df_pendientes["id_recorte"].isin(revisados)
].reset_index(drop=True)

# ------------------------------------------------------------
# 3. Crear interfaz interactiva
# ------------------------------------------------------------

tamano_lote = 10
salida_lote = widgets.Output()
estado_general = widgets.HTML()

boton_guardar = widgets.Button(
    description="Guardar lote y continuar",
    button_style="success",
    icon="save",
    layout=widgets.Layout(width="220px")
)

filas_interfaz = []

# ------------------------------------------------------------
# 4. Mostrar lote de matrículas
# ------------------------------------------------------------

def mostrar_lote():

    global filas_interfaz

    filas_interfaz = []

    with salida_lote:
        clear_output(wait=True)

        if df_pendientes.empty:
            print("VALIDACIÓN FINALIZADA")
            print("Registros revisados:", len(validacion_lotes))
            print("Archivo:", ruta_validacion)
            return

        lote = df_pendientes.head(tamano_lote)

        estado_general.value = (
            f"<b>Revisados:</b> {len(validacion_lotes)} "
            f"| <b>Pendientes:</b> {len(df_pendientes)} "
            f"| <b>Lote actual:</b> {len(lote)}"
        )

        print("=" * 55)
        print("REVISIÓN MANUAL DE MATRÍCULAS")
        print("=" * 55)

        for _, fila in lote.iterrows():

            clave = fila["id_recorte"]
            dato = recortes_validos[clave]

            titulo = widgets.HTML(
                value=(
                    f"<b>{clave}</b> "
                    f"| Prioridad: {fila['prioridad']}"
                )
            )

            imagen = widgets.Image(
                value=__import__("io").BytesIO(
                    _imagen_png(dato["imagen"])
                ).getvalue(),
                format="png",
                width=300
            )

            etiqueta = widgets.Text(
                value=str(dato["etiqueta"]),
                description="Matrícula:",
                style={"description_width": "100px"},
                layout=widgets.Layout(width="350px")
            )

            decision = widgets.Dropdown(
                options=[
                    ("Pendiente", "pendiente"),
                    ("Validar", "validada"),
                    ("Corregir", "corregida"),
                    ("Descartar", "descartada")
                ],
                value="pendiente",
                description="Decisión:",
                style={"description_width": "100px"},
                layout=widgets.Layout(width="250px")
            )

            observacion = widgets.Text(
                placeholder="Observación opcional",
                layout=widgets.Layout(width="450px")
            )

            filas_interfaz.append({
                "clave": clave,
                "dato": dato,
                "etiqueta": etiqueta,
                "decision": decision,
                "observacion": observacion
            })

            display(titulo)
            display(imagen)
            display(etiqueta)
            display(decision)
            display(observacion)
            display(widgets.HTML("<hr>"))

        display(boton_guardar)

# ------------------------------------------------------------
# 5. Función auxiliar para visualizar imágenes
# ------------------------------------------------------------

def _imagen_png(imagen):
    import io
    buffer = io.BytesIO()
    imagen.convert("RGB").save(buffer, format="PNG")
    return buffer.getvalue()

# ------------------------------------------------------------
# 6. Guardar decisiones del lote
# ------------------------------------------------------------

def guardar_lote(_):

    global validacion_lotes
    global df_pendientes

    nuevos = []

    for elemento in filas_interfaz:

        estado = elemento["decision"].value

        if estado == "pendiente":
            continue

        dato = elemento["dato"]
        etiqueta = elemento["etiqueta"].value.strip().upper()
        original = str(dato["etiqueta"]).strip().upper()

        # Controlar consistencia de las decisiones.
        if estado == "validada" and etiqueta != original:
            print(
                "Revisar:", elemento["clave"],
                "La etiqueta cambió; selecciona Corregir."
            )
            return

        if estado in ("validada", "corregida") and not etiqueta:
            print(
                "Revisar:", elemento["clave"],
                "La matrícula no puede estar vacía."
            )
            return

        if estado == "corregida" and etiqueta == original:
            print(
                "Revisar:", elemento["clave"],
                "La transcripción no cambió; selecciona Validar."
            )
            return

        nuevos.append({
            "id_recorte": elemento["clave"],
            "id_imagen": dato["id"],
            "region": dato["region"],
            "etiqueta_original": dato["etiqueta"],
            "etiqueta_verificada": (
                etiqueta if estado != "descartada" else ""
            ),
            "estado": estado,
            "observaciones": elemento["observacion"].value
        })

    if not nuevos:
        print("No hay decisiones para guardar.")
        return

    validacion_lotes = pd.concat(
        [validacion_lotes, pd.DataFrame(nuevos)],
        ignore_index=True
    )

    validacion_lotes = validacion_lotes.drop_duplicates(
        subset=["id_recorte"],
        keep="last"
    )

    validacion_lotes.to_csv(
        ruta_validacion,
        index=False,
        encoding="utf-8-sig"
    )

    completados = {r["id_recorte"] for r in nuevos}

    df_pendientes = df_pendientes[
        ~df_pendientes["id_recorte"].isin(completados)
    ].reset_index(drop=True)

    mostrar_lote()

# ------------------------------------------------------------
# 7. Activar botones y mostrar primer lote
# ------------------------------------------------------------

boton_guardar.on_click(guardar_lote)

display(estado_general)
display(salida_lote)

mostrar_lote()

print("Interfaz de revisión por lotes preparada.")


HTML(value='')

Output()

Interfaz de revisión por lotes preparada.


In [16]:
# CELDA 15A — Mostrar botón de guardado fuera de la lista
display(boton_guardar)

Button(button_style='success', description='Guardar lote y continuar', icon='save', layout=Layout(width='220px…

Revisar: sonne_00020_region_0 La transcripción no cambió; selecciona Validar.
Revisar: sonne_00020_region_0 La transcripción no cambió; selecciona Validar.
